# 09. Validación de las etiquetas humanas

Este cuaderno revisa la clasificación asistida de las publicaciones destinadas al ajuste. La persona observó la sugerencia de M0 antes de asignar o confirmar la etiqueta. En cambio, las 100 publicaciones de prueba se etiquetaron sin observar predicciones y permanecen separadas.

## A. Abrir los archivos

- **Qué hace:** carga la clasificación asistida, el conjunto preparado y la prueba independiente.
- **Para qué:** comprobar identificadores, columnas y etiquetas antes del entrenamiento.
- **Por qué:** el modelado necesita una referencia humana válida y sin cruces con la prueba.

In [ ]:
%pip install -q openpyxl matplotlib scikit-learn

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.metrics import confusion_matrix

if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/TESIS_DATA_SCIENCE/COLAB')
else:
    RAIZ = Path.cwd()

if not (RAIZ / 'TABLAS').is_dir():
    raise FileNotFoundError('Abre este cuaderno desde la carpeta COLAB o ajusta RAIZ.')

CLASIFICACION = RAIZ / 'TABLAS/06_clasificacion/clasificacion_asistida_v5_etiquetada.xlsx'
DATASET = RAIZ / 'TABLAS/05_preparacion/dataset_preparado_v4.xlsx'
PRUEBA = RAIZ / 'TABLAS/05_preparacion/muestra_prueba_100_v4.xlsx'
SALIDA = RAIZ / 'TABLAS/09_etiquetas'
SALIDA.mkdir(parents=True, exist_ok=True)

for ruta in [CLASIFICACION, DATASET, PRUEBA]:
    if not ruta.is_file():
        raise FileNotFoundError(f'Falta el archivo requerido: {ruta.name}')

anotadas = pd.read_excel(CLASIFICACION, sheet_name='clasificacion', dtype=object, keep_default_na=False)
dataset = pd.read_excel(DATASET, dtype=object, keep_default_na=False)
prueba = pd.read_excel(PRUEBA, sheet_name='evaluacion', dtype=object, keep_default_na=False)
display(anotadas.head(10))

## B. Comprobar identificadores y etiquetas

- **Qué hace:** valida unicidad, categorías permitidas y separación de la prueba.
- **Para qué:** demostrar que las filas anotadas corresponden exactamente al conjunto de ajuste.
- **Por qué:** una publicación de prueba dentro del ajuste produciría fuga de información.

In [ ]:
REQUERIDAS = {
    'id', 'fuente', 'texto_real', 'texto_procesado',
    'polaridad_sugerida', 'etiqueta_humana', 'observacion_humana',
}
VALIDAS = ['negativa', 'neutral', 'positiva']
PERMITIDAS = set(VALIDAS + ['inválida'])

faltantes = REQUERIDAS - set(anotadas.columns)
if faltantes:
    raise ValueError(f'Faltan columnas: {sorted(faltantes)}')

for tabla in [anotadas, dataset, prueba]:
    tabla['id'] = pd.to_numeric(tabla['id'], errors='raise').astype(int)
anotadas['etiqueta_humana'] = anotadas['etiqueta_humana'].astype(str).str.strip().str.lower()
anotadas['polaridad_sugerida'] = anotadas['polaridad_sugerida'].astype(str).str.strip().str.lower()
prueba['etiqueta_polaridad'] = prueba['etiqueta_polaridad'].astype(str).str.strip().str.lower()

if anotadas['id'].duplicated().any():
    raise ValueError('La clasificación contiene IDs duplicados.')
no_permitidas = sorted(set(anotadas['etiqueta_humana']) - PERMITIDAS)
if no_permitidas:
    raise ValueError(f'Existen etiquetas no permitidas: {no_permitidas}')
if not prueba['etiqueta_polaridad'].isin(VALIDAS).all():
    raise ValueError('La prueba contiene etiquetas fuera de negativa, neutral y positiva.')
if set(anotadas['id']) & set(prueba['id']):
    raise ValueError('Los IDs de ajuste se cruzan con la prueba independiente.')

ids_esperados = set(dataset['id']) - set(prueba['id'])
if set(anotadas['id']) != ids_esperados:
    faltan = len(ids_esperados - set(anotadas['id']))
    sobran = len(set(anotadas['id']) - ids_esperados)
    raise ValueError(f'Los IDs no coinciden con dataset - prueba. Faltan {faltan}; sobran {sobran}.')

validas = anotadas[anotadas['etiqueta_humana'].isin(VALIDAS)].copy()
invalidas = anotadas[anotadas['etiqueta_humana'] == 'inválida'].copy()
acuerdo = float((validas['etiqueta_humana'] == validas['polaridad_sugerida']).mean())

controles = pd.DataFrame([
    {'control': 'IDs anotados únicos', 'resultado': anotadas['id'].nunique()},
    {'control': 'Cruce con prueba', 'resultado': len(set(anotadas.id) & set(prueba.id))},
    {'control': 'IDs esperados', 'resultado': len(ids_esperados)},
    {'control': 'IDs anotados', 'resultado': len(anotadas)},
])
display(controles)
display(invalidas[['id', 'fuente', 'observacion_humana']].head(10))

## C. Describir la clasificación

- **Qué hace:** cuenta clases, mide la coincidencia con M0 y cruza la sugerencia con la etiqueta humana.
- **Para qué:** documentar la composición del corpus que recibe el entrenamiento.
- **Por qué:** el acuerdo con una sugerencia automática no equivale a acuerdo entre dos personas.

In [ ]:
conteo_sugerido = anotadas['polaridad_sugerida'].value_counts().reindex(VALIDAS, fill_value=0)
conteo_humano = validas['etiqueta_humana'].value_counts().reindex(VALIDAS, fill_value=0)
conteo_prueba = prueba['etiqueta_polaridad'].value_counts().reindex(VALIDAS, fill_value=0)

distribuciones = pd.DataFrame({
    'sugerida_M0': conteo_sugerido,
    'humana_valida': conteo_humano,
    'prueba_ciega': conteo_prueba,
})
distribuciones.index.name = 'clase'

matriz = confusion_matrix(
    validas['etiqueta_humana'], validas['polaridad_sugerida'], labels=VALIDAS
)
matriz_df = pd.DataFrame(matriz, index=VALIDAS, columns=VALIDAS)
matriz_df.index.name = 'etiqueta_humana'
matriz_df.columns.name = 'sugerencia_M0'

resumen = {
    'registros_anotados': int(len(anotadas)),
    'registros_validos': int(len(validas)),
    'registros_invalidos': int(len(invalidas)),
    'distribucion_sugerida': {k: int(v) for k, v in conteo_sugerido.items()},
    'distribucion_humana_valida': {k: int(v) for k, v in conteo_humano.items()},
    'distribucion_prueba_ciega': {k: int(v) for k, v in conteo_prueba.items()},
    'acuerdo_sugerencia_humano': acuerdo,
    'tipo_etiquetado_ajuste': 'asistido; la sugerencia de M0 era visible',
    'tipo_etiquetado_prueba': 'sin sugerencia automática',
}

display(pd.DataFrame([{
    'anotadas': len(anotadas), 'válidas': len(validas), 'inválidas': len(invalidas),
    'acuerdo_sugerencia_humano': acuerdo,
}]))
display(distribuciones)
display(matriz_df)

## D. Guardar la evidencia

- **Qué hace:** exporta el resumen, las filas inválidas sin texto y las distribuciones.
- **Para qué:** conectar esta validación con los cuadernos 10 y 11.
- **Por qué:** las cifras del informe deben proceder de archivos identificables.

In [ ]:
(SALIDA / 'resumen_etiquetas.json').write_text(
    json.dumps(resumen, ensure_ascii=False, indent=2), encoding='utf-8'
)
invalidas[['id', 'fuente', 'observacion_humana']].to_excel(
    SALIDA / 'registros_invalidos.xlsx', index=False
)
with pd.ExcelWriter(SALIDA / 'distribuciones_etiquetas.xlsx', engine='openpyxl') as escritor:
    distribuciones.reset_index().to_excel(escritor, sheet_name='distribuciones', index=False)
    matriz_df.to_excel(escritor, sheet_name='matriz_sugerida_humana')

figura, eje = plt.subplots(figsize=(9, 5))
distribuciones.plot(kind='bar', ax=eje, color=['#4C78A8', '#F58518', '#54A24B'])
eje.set_title('Distribución de las etiquetas')
eje.set_xlabel('Polaridad')
eje.set_ylabel('Publicaciones')
eje.tick_params(axis='x', rotation=0)
eje.legend(['Sugerida por M0', 'Humana válida', 'Prueba sin sugerencia'])
eje.spines[['top', 'right']].set_visible(False)
figura.tight_layout()
figura.savefig(SALIDA / 'distribucion_etiquetas.png', dpi=170, bbox_inches='tight')
plt.show()

print('Archivos guardados en:', SALIDA)
display(invalidas[['id', 'fuente', 'observacion_humana']].head(10))